# 22 · Fault handling and NULL behaviour (paper §14)
Code: `VAPMIP/rotary_rerun_monad.py` (`Fault`, `Status`, `Relation`, `Ledger`, `Harness`, `MindsEye`), `ValaQuenta/modules/box_kite/maths.py` (`fixed_point_weight`), `VAPMIP/PtolC/ptol.c` (`measure_sigma`).
Part 1: every fault is a record. Part 2: what each program returns for an all-zero input (a census; each entry is a data point).

In [1]:
from pn import *
L = Ledger("22 · fault handling and NULL behaviour")
with quiet():
    import rotary_rerun_monad as rr
    h = rr.Harness(); h.run(psi=rr._demo_psi(), currents=(0.412, -0.187, -0.201))
led = h.ledger
counts = {s.name: led.count(s) for s in rr.Status}
print("records:", len(led), counts)
print("faults by kind:", {k.name: len(led.faults_of(k)) for k in rr.Fault})
for r in led.faults()[:3]: print(f"  [{r.fault.name}] {r.name}: expected {r.expected!r}, observed {r.observed!r}")
L.check("every check is a record (none is a bare count)", len(led) == sum(counts.values()) and len(led) > 50, f"{len(led)} records")
L.check("a violated record is classified MATHS or CODE, never left unclassified", all(r.fault is not rr.Fault.NONE for r in led.faults()))
L.check("the deliberately unbalanced currents produce at least one fault for the harness to find", len(led.faults()) >= 1, f"{len(led.faults())} faults")
print("maths_works:", led.maths_works)

records: 124 {'HOLDS': 122, 'VIOLATED': 1, 'DEGENERATE': 1, 'UNTESTED': 0}
faults by kind: {'MATHS': 1, 'CODE': 0, 'NONE': 0}
  [MATHS] current.neutral: expected 0.0, observed 0.023999999999999966
maths_works: False


## NULL census: an all-zero input

In [2]:
with quiet():
    from ValaQuenta.modules.box_kite import maths as bk
rows = []
def row(name, ret, keeps): rows.append((name, ret, keeps)); print(f"  {'KEEPS NULL ' if keeps else 'SUPPLIES   '} {name:62s} -> {ret}")
snap = rr.MindsEye().snapshot([0.0] * 16)["sigma_self"]
row("rotary_rerun_monad.MindsEye.snapshot(zeros)['sigma_self']", repr(snap), isinstance(snap, float) and math.isnan(snap))
row("box_kite.fixed_point_weight(zeros)", repr(bk.fixed_point_weight([0.0] * 16)), False)
out, err, _ = ptol("-r", "")
sig = next((l.split("σ_self:")[1].split()[0] for l in err.splitlines() if "σ_self" in l), "?")
row('ptol -r ""  (σ_self printed)', sig, sig.lower() == "nan")
src = open(os.path.join(VAPMIP, "PtolC", "ptol.c")).read()
row("ptol.c measure_sigma: `if (total < 1e-15) return 0.5;`", "returns 0.5 for a zero vector" if "if (total < 1e-15) return 0.5;" in src else "(line not found)", False)
with quiet(): import ptolemy_console as pc; ad = pc.ScaledMindEyeMonadAdapter()
with quiet(): rep, meta = ad.say("good morning")
row("ScaledMindEyeMonadAdapter.say('good morning') (no verb resolves)", rep[:70], rep.startswith("(19D"))
L.check("census recorded (not a verdict on the engines)", True, f"{sum(1 for r in rows if r[2])} of {len(rows)} paths keep NULL; {sum(1 for r in rows if not r[2])} supply a value")

  KEEPS NULL  rotary_rerun_monad.MindsEye.snapshot(zeros)['sigma_self']      -> nan
  SUPPLIES    box_kite.fixed_point_weight(zeros)                             -> 0.0
  SUPPLIES    ptol -r ""  (σ_self printed)                                   -> 0.5000000000
  SUPPLIES    ptol.c measure_sigma: `if (total < 1e-15) return 0.5;`         -> returns 0.5 for a zero vector


  KEEPS NULL  ScaledMindEyeMonadAdapter.say('good morning') (no verb resolves) -> (19D: nothing resolved — no WordNet verb found, 2 content word(s) reso


True

## Ledger

In [3]:
_n = L.show()

LEDGER — 22 · fault handling and NULL behaviour
  [PASS ] every check is a record (none is a bare count)
          124 records
  [PASS ] a violated record is classified MATHS or CODE, never left unclassified
  [PASS ] the deliberately unbalanced currents produce at least one fault for the harness to find
          1 faults
  [PASS ] census recorded (not a verdict on the engines)
          2 of 5 paths keep NULL; 3 supply a value
  -> PASS 4 · FAIL 0 · PROSE 0
